# Loading Dependencies

In [ ]:
import torch, os
from tqdm.auto import tqdm
from fastdownload import FastDownload
from fastcore.all import concat
from PIL import Image
from kaggle_secrets import UserSecretsClient
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionImg2ImgPipeline,
)

# Set device

In [ ]:
if torch.cuda.is_available():
  torch_device = "cuda"
elif torch.backends.mps.is_available():
  torch_device = "mps"
  os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = "1"
else: torch_device = "cpu"

# HF Login

In [ ]:
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
!hf auth whoami

# Inference with stable diffusion pipeline

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    variant = "fp16",
    device_map=torch_device
)

In [ ]:
# The weights are cached here by default.
os.listdir("/root/.cache/huggingface/hub")

Now, we are ready to prompt the model
- If your GPU is not big enough to use `pipe`, run `pipe.enable_attention_slicing()`
- As described in the docs:
    - When this option is enabled, the attention module will
    - split the input tensor into slices to compute attention
    - in several steps. This is useful to save some memory
    - in exchange for a small speed decrease.

In [ ]:
prompt = "a photograph of an astronaut riding a horse"
torch.manual_seed(1)
pipe(prompt).images[0]

In [ ]:
torch.manual_seed(1024)
pipe(prompt).images[0]

You will have noticed that running the pipeline shows a progress bar with a certain number of steps. This is because Stable Diffusion is based on a progressive denoising algorithm that is able to create a convincing image starting from pure random noise. Models in this family are known as _diffusion models_.

In [ ]:
torch.manual_seed(1024)
pipe(prompt, num_inference_steps=3).images[0]

In [ ]:
torch.manual_seed(1024)
pipe(prompt, num_inference_steps=16).images[0]

## Classifier-Free Guidance

- _Classifier-Free Guidance_ is a method to increase the adherence of the output to the conditioning signal we used (the text).
- Roughly speaking, the larger the guidance the more the model tries to represent the text prompt.
- However, large values tend to produce less diversity. The default is `7.5`, which represents a good compromise between variety and fidelity.
- This [blog post](https://benanne.github.io/2022/05/26/guidance.html) goes into deeper details on how it works.

In [ ]:
def image_grid(imgs, rows, cols):
    w,h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    for i, img in enumerate(imgs): grid.paste(img, box=(i%cols*w, i//cols*h))
    return grid

- We can generate multiple images for the same prompt by simply passing a list of prompts instead of a string.

In [ ]:
num_rows,num_cols = 4,4
prompts = [prompt] * num_cols
images = concat(pipe(prompts, guidance_scale=g).images for g in [1.1,3,7,14])
image_grid(images, rows=num_rows, cols=num_cols)

## Negative prompts

In [ ]:
torch.manual_seed(1000)
prompt = "Labrador in the style of Vermeer"
img0 = pipe(prompt).images[0]

torch.manual_seed(1000)
img1 = pipe(prompt, negative_prompt="blue").images[0]

image_grid([img0, img1], rows=1, cols=2)

## Image to Image

- Even though Stable Diffusion was trained to generate images, and optionally drive the generation using text conditioning, we can use the raw image diffusion process for other tasks.
- For example, instead of starting from pure noise, we can start from an image an add a certain amount of noise to it. We are replacing the initial steps of the denoising and pretending our image is what the algorithm came up with.
- Then we continue the diffusion process from that state as usual. This usually preserves the composition although details may change a lot. It's great for sketches!
- These operations (provide an initial image, add some noise to it and run diffusion from there) can be automatically performed by a special image to image pipeline called `StableDiffusionImg2ImgPipeline`.
- We'll use as an example the following sketch created by [user VigilanteRogue81](https://huggingface.co/spaces/huggingface-projects/diffuse-the-rest/discussions/204).

In [ ]:
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    variant="fp16",
    dtype=torch.bfloat16,
    device_map = torch_device
)

In [ ]:
fast_downloader = FastDownload()
hf_cdn = 'https://cdn-uploads.huggingface.co'
sketch_path = f'{hf_cdn}/production/uploads/1664665907257-noauth.png'
p = fast_downloader.download(sketch_path)
init_image = Image.open(p).convert("RGB")
display(init_image)

- `image` is used as a starting point, and more noise is added the higher the `strength`.
- The number of denoising steps depends on the amount of noise initially added.
- When `strength` is 1, added noise is maximum, and the denoising process runs for the full number of iterations specified in `num_inference_steps`.
- A value of 1 essentially ignores `image`.

In [ ]:
torch.manual_seed(1000)
prompt = "Wolf howling at the moon, photorealistic 4K"
images = pipe(prompt=prompt,
              num_images_per_prompt=3,
              image=init_image,
              strength=0.8,
              num_inference_steps=50).images
image_grid(images, rows=1, cols=3)

- When we get a composition we like, we can use it as the next seed for another prompt and further change the results.
- For example, let's take the third image above and try to use it to generate something in the style of Van Gogh.

In [ ]:
init_image = images[2]
torch.manual_seed(1000)
prompt = "Oil painting of wolf howling at the moon by Van Gogh"
images = pipe(prompt=prompt,
              num_images_per_prompt=3,
              image=init_image,
              strength=1,
              num_inference_steps=70).images
image_grid(images, rows=1, cols=3)

## Textual inversion

- Textual inversion is a process where you can quickly "teach" a new word to the text model and train its embeddings close to some visual representation.
- This is achieved by
    - adding a new token to the vocabulary,
    - freezing the weights of all the models (except the text encoder),
    - and train with a few representative images.

This is a schematic representation of the process by the [authors of the paper](https://textual-inversion.github.io).

<img src="https://textual-inversion.github.io/static/images/training/training.JPG" alt="Textual Inversion diagram" width="700" height="700">

- You can train your own tokens with photos you provide using [this training script](https://github.com/huggingface/diffusers/tree/main/examples/textual_inversion) or [Google Colab notebook](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/sd_textual_inversion_training.ipynb).
- There's also a [Colab notebook for inference](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/stable_conceptualizer_inference.ipynb)
    - but we'll show below the steps we have to follow to add a trained token to the vocabulary and make it work the pre-trained Stable Diffusion model.
    - We'll try an example using embeddings trained for [this style](https://huggingface.co/sd-concepts-library/indian-watercolor-portraits).

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    variant="fp16",
    torch_dtype=torch.bfloat16,
    device_map = torch_device
)

styles_repo = "https://huggingface.co/sd-concepts-library"
style_name = "indian-watercolor-portraits"
embeds_url = f"{styles_repo}/{style_name}/resolve/main/learned_embeds.bin"
embeds_path = fast_downloader.download(embeds_url)
embeds_dict = torch.load(str(embeds_path), map_location="cpu")

- The embeddings for the new token are stored in a small PyTorch pickled dictionary, whose key is the new text token that was trained.
- Since the encoder of our pipeline does not know about this term, we need to manually append it.
- We add the new token to the tokenizer and the trained embeddings to the embeddings table.

In [ ]:
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder
new_token, embeds = next(iter(embeds_dict.items()))
embeds = embeds.to(text_encoder.dtype)
print(new_token)
print(embeds.shape)

In [ ]:
assert tokenizer.add_tokens(new_token) == 1, "The token already exists!"
text_encoder.resize_token_embeddings(len(tokenizer))
new_token_id = tokenizer.convert_tokens_to_ids(new_token)
text_encoder.get_input_embeddings().weight.data[new_token_id] = embeds

We can now run inference and refer to the style as if it was another word in the dictionary.

In [ ]:
torch.manual_seed(1000)
image = pipe(f"Woman reading in the style of {new_token}").images[0]
image

## Dreambooth

- [Dreambooth](https://dreambooth.github.io) is a kind of fine-tuning that attempts to introduce new subjects by providing just a few images of the new subject.
- The goal is similar to that of [Textual Inversion](#Textual-Inversion), but the process is different.
- Instead of creating a new token as Textual Inversion does, we select an existing token in the vocabulary (usually a rarely used one), and fine-tune the model for a few hundred steps to bring that token close to the images we provide.
- This is a regular fine-tuning process in which all modules are **unfrozen**.
- For example, we fine-tuned a model with a prompt like `"photo of a sks person"`, using the rare `sks` token to qualify the term `person`, and using photos of [Jeremy](https://www.kaggle.com/jhoward) as the targets. Let's see how it works.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "pcuenq/jh_dreambooth_1000",
    torch_dtype=torch.bfloat16,
    device_map = torch_device
)
torch.manual_seed(1000)
prompt = "Painting of sks person in the style of Paul Signac"
images = pipe(prompt, num_images_per_prompt=4).images
image_grid(images, 1, 4)

- Fine-tuning with Dreambooth is finicky and sensitive to hyperparameters, as we are essentially asking the model to overfit the prompt to the supplied images.
- In some situations, we observe problems such as catastrophic forgetting of the associated term (`"person"` in this case).
- The authors applied a technique called "prior preservation" to try to avoid it by performing a special regularization using other examples of the term, in addition to the ones we provided.
- The cool thing about this idea is that those examples can be easily generated by the pre-trained Stable Diffusion model itself!
- This method is not used to train the model used in the previous example.
- Other ideas that may work include
    - use EMA so that the final weights preserve some of the previous knowledge
    - use progressive learning rates for fine-tuning,
    - combine the best of Textual Inversion with Dreambooth

- If you want to train your own Dreambooth model, you can use [this script](https://github.com/huggingface/diffusers/tree/main/examples/dreambooth) or [this Colab notebook](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/sd_dreambooth_training.ipynb).

# What is stable diffusion

**Stable Diffusion** is based on a particular type of diffusion model called **Latent Diffusion**, proposed in [High-Resolution Image Synthesis with Latent Diffusion Models](https://arxiv.org/abs/2112.10752).

**General diffusion** models are machine learning systems that are trained to *denoise* random gaussian noise step by step, to get to a sample of interest, such as an *image*.
- Diffusion models have shown to achieve state-of-the-art results for generating image data.
- But one **downside** of diffusion models is that the reverse denoising process is __slow__.
- In addition, these models consume __a lot of memory__ because they operate in pixel space, which becomes unreasonably expensive when generating high-resolution images.
- Therefore, it is challenging to train these models and also use them for inference.

**Latent diffusion** can reduce the memory and compute complexity by applying the diffusion process over a __lower dimensional _latent_ space__, instead of using the actual pixel space. This is the key difference between standard diffusion and latent diffusion models: **in latent diffusion the model is trained to generate latent (compressed) representations of the images.**

There are three main components in latent diffusion.

1. An autoencoder (VAE).
2. A [U-Net](https://colab.research.google.com/github/huggingface/notebooks/blob/main/diffusers/diffusers_intro.ipynb#scrollTo=wW8o1Wp0zRkq).
3. A text-encoder, *e.g.* [CLIP's Text Encoder](https://huggingface.co/docs/transformers/model_doc/clip#transformers.CLIPTextModel).

The output of the U-Net, being the noise residual, is used to compute a denoised latent image representation via a scheduler algorithm. Many different scheduler algorithms can be used for this computation, each having its pros and cons. For Stable Diffusion, we recommend using one of:

- [PNDM scheduler](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_pndm.py) (used by default)
- [DDIM scheduler](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_ddim.py)
- [K-LMS scheduler](https://github.com/huggingface/diffusers/blob/main/src/diffusers/schedulers/scheduling_lms_discrete.py)

**Why is latent diffusion fast and efficient?**

- Since the U-Net of latent diffusion models operates on a low dimensional space, it greatly reduces the memory and compute requirements compared to pixel-space diffusion models.
- For example, the autoencoder used in Stable Diffusion has a reduction factor of 8 but uses 4 channels instead of 3. This means that an image of shape `(3, 512, 512)` becomes `(4, 64, 64)` in latent space, which requires `8 × 8 × 3/4 = 48` times less memory.
- This is why it's possible to generate `512 × 512` images so quickly.

## Latents and `callback`

The Stable Diffusion pipeline can send intermediate latents to a `callback` function we provide. By running these latents through the image decoder (the `vae` component of the pipeline), we can see how the denoising process progresses and the image unfolds.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "pcuenq/jh_dreambooth_1000",
    device_map = torch_device
)
images = []

In [ ]:
images = []
call_back_step = 12
# run ??pipe.__call__ to see how to implement this
def latents_callback(pipe, i, t, kwargs):
    if i%call_back_step == 0:
        print(f"inside the callback function")
        latents = kwargs["latents"]
        print(f"i: {i} | t: {t} | latents shape: {latents.shape}")
        latents = 1 / 0.18215 * latents # this is based on the paper
        vae = pipe.vae
        image = vae.decode(latents).sample[0]
        image = (image / 2 + 0.5).clamp(0, 1)
        image = image.cpu().permute(1, 2, 0).numpy()
        images.extend(pipe.numpy_to_pil(image))
    return {} # or kwargs

In [ ]:
prompt = "Portrait painting of Jeremy Howard looking happy."
torch.manual_seed(9000)
final_image = pipe(
    prompt,
    callback_on_step_end=latents_callback,
    callback_on_step_end_tensor_inputs=["latents"], # this is the default value
).images[0]
images.append(final_image)
image_grid(images, rows=1, cols=len(images))